In [1]:
import pandas as pd

data = pd.read_csv('FBlocation/train.csv')
# data = data.dropna() # 经过去空结果可以发现，该数据集是没有null值的
print(data.shape)
data.head()

(29118021, 6)


,row_id,x,y,accuracy,time,place_id
0,0,0.7941,9.0809,54,470702,8523065625
1,1,5.9567,4.7968,13,186555,1757726713
2,2,8.3078,7.0407,74,322648,1137537235
3,3,7.3665,2.5165,65,704587,6567393236
4,4,4.0961,1.1307,31,472130,7440663949


In [2]:
import numpy as np

y = data['place_id']
x = data[['x','y','accuracy','time']].copy()

x['time'] = pd.to_datetime(x['time'] , unit='s')
x['day'] = x['time'].dt.day
x['hour'] = x['time'].dt.hour
x['weekday'] = x['time'].dt.weekday

# ⭐ 时间特征增强（重点）
x['hour_sin'] = np.sin(2 * np.pi * x['hour'] / 24)
x['hour_cos'] = np.cos(2 * np.pi * x['hour'] / 24)

# （可选进阶）
x['weekday_sin'] = np.sin(2 * np.pi * x['weekday'] / 7)
x['weekday_cos'] = np.cos(2 * np.pi * x['weekday'] / 7)


# 这一步的作用是将x和y分别进行网格化（乘以10并取整），这样可以将地理位置的连续特征转换为离散的网格编号，有助于模型捕捉空间分布的规律，提高算法的空间敏感性，尤其适用于地理定位任务（如KNN）。即“空间分桶/分箱”。
x['x_grid'] = (x['x'] * 10).astype(int)
x['y_grid'] = (x['y'] * 10).astype(int)

x = x.drop('time',axis=1)
x.head()

,x,y,accuracy,day,hour,weekday,hour_sin,hour_cos,weekday_sin,weekday_cos,x_grid,y_grid
0,0.7941,9.0809,54,6,10,1,0.500000,-0.866025,0.781831,0.623490,7,90
1,5.9567,4.7968,13,3,3,5,0.707107,0.707107,-0.974928,-0.222521,59,47
2,8.3078,7.0407,74,4,17,6,-0.965926,-0.258819,-0.781831,0.623490,83,70
3,7.3665,2.5165,65,9,3,4,0.707107,0.707107,-0.433884,-0.900969,73,25
4,4.0961,1.1307,31,6,11,1,0.258819,-0.965926,0.781831,0.623490,40,11


In [3]:
unique_places = y.nunique()
print(f"数据集中有{unique_places}个不同的地点")
place_counts = y.value_counts()
mask = y.isin(place_counts[place_counts > 22].index)
x = x[mask]
y = y[mask]

数据集中有108390个不同的地点


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# n_neighbors=25: 设置KNN算法查找最近邻时，选取距离待预测点最近的25个邻居作为参考。
# weights='distance': 表示每个邻居对预测结果的贡献按距离加权，越近的邻居权重越大，预测结果更受近邻影响。
# metric='manhattan': 距离的计算方式为曼哈顿距离，即各特征维度差值的绝对值之和（L1范数）。
knn = KNeighborsClassifier(
    n_neighbors=25,
    weights='distance',
    metric='manhattan'
)
knn.fit(X_train,y_train)

# 找邻居
distances, indices = knn.kneighbors(X_test)

# ⭐ 正确写法
neighbors_labels = y_train.values[indices]

from collections import Counter
import numpy as np

top3_preds = []

for row in neighbors_labels:
    counter = Counter(row)
    top3 = [item[0] for item in counter.most_common(3)]
    top3_preds.append(top3)

# 计算Top3
score = np.mean([
    y_test.iloc[i] in top3_preds[i]
    for i in range(len(y_test))
])

print("Top-3 accuracy:", score)

# y_pred_proba = knn.predict_proba(X_test)
# top3 = np.argsort(y_pred_proba, axis=1)[:, -3:]

# score = np.mean([
#     y_test.iloc[i] in knn.classes_[top3[i]]
#     for i in range(len(y_test))
# ])

# print("Top-3 accuracy:", score)

# accuracy = knn.score(X_test,y_test)
# print(f"KNN模型的准确率为：{accuracy:.2f}")

Top-3 accuracy: 0.2855695450774289
